In [ ]:
import gdown
import zipfile
import os

# Download the zip from Google Drive
file_id = "1ijdOswMou-azg3Y5CLPQX76iFILSrzIN"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "FullDataset.zip", quiet=False, fuzzy=True)

# Extract the zip
with zipfile.ZipFile("FullDataset.zip", "r") as zip_ref:
    zip_ref.extractall("FullDataset/")

print(os.listdir("FullDataset/"))


In [ ]:
# #This module uses a VGG16 CNN to classify white blood cell types

import os
import numpy as np
from pathlib import Path
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import sys


from model import VGG16
from train import train_one_epoch, validate
from dataset import get_loaders
import gdown


# =========================================================
# 1. Settings
# =========================================================
#gdown.download_folder(
 #   url="https://drive.google.com/drive/folders/YOUR_FOLDER_ID",
 #   output="dataset_5classes",
 #   quiet=False
#)

dataset_dir = Path.cwd() / "dataset_5classes/train"



# Training settings
batch_size = 32
num_epochs = 10
learning_rate = 1e-4
image_size = 224
val_split = 0.2
random_seed = 42

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


#==================================================
# 2. Load the Dataset!
#==================================================
train_loader, val_loader, class_names, num_classes = get_loaders(
    dataset_dir, batch_size, image_size, val_split, random_seed
)

print(f"Train images: {len(train_loader.dataset)}")
print(f"Validation images: {len(val_loader.dataset)}")

# =========================================================
# 3. VGG16 Model
# =========================================================
model = VGG16(num_classes=num_classes).to(device)
print(model)


# =========================================================
# 4. Loss and optimizer
# =========================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

#print("ENDING HERE")
#sys.exit()
# =========================================================
# 5. Training loop
# =========================================================
best_val_acc = 0.0


for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_vgg16.pth")
        print("Saved best model to best_vgg16.pth")


# =========================================================
# 6. Save final model
# =========================================================
torch.save(model.state_dict(), "final_vgg16.pth")
print("Training complete.")
print("Saved final model to final_vgg16.pth")